# Merge Upper Plate Datasets

รวม 2 datasets:
1. `plate_upper_synth_special` - Synthetic special plates (50K)
2. `upper_train` - Real upper plates (158K)

Output: `upper_train_with_special` - Combined dataset (~208K samples)

In [6]:
import os
import sys
import shutil
from pathlib import Path
import pandas as pd
from tqdm import tqdm

# Paths
PROJECT_ROOT = Path(r"d:\CodingD\ALPR")

# Source datasets
SYNTH_SPECIAL_DIR = PROJECT_ROOT / "data" / "plate_upper_synth_special"
SYNTH_SPECIAL_DATA = SYNTH_SPECIAL_DIR / "data"
SYNTH_SPECIAL_CSV = SYNTH_SPECIAL_DIR / "labels.csv"

UPPER_TRAIN_DIR = PROJECT_ROOT / "train_ocr" / "data" / "upper_train"
UPPER_TRAIN_DATA = UPPER_TRAIN_DIR / "data"
UPPER_TRAIN_CSV = UPPER_TRAIN_DIR / "labels.csv"

# Output (merged dataset)
OUTPUT_DIR = PROJECT_ROOT / "train_ocr" / "data" / "upper_train_with_special"
OUTPUT_DATA = OUTPUT_DIR / "data"
OUTPUT_CSV = OUTPUT_DIR / "labels.csv"

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DATA.mkdir(parents=True, exist_ok=True)

print("Project Root:", PROJECT_ROOT)
print("\nSource 1 (Synthetic Special):", SYNTH_SPECIAL_DIR)
print("Source 2 (Upper Train):", UPPER_TRAIN_DIR)
print("\nOutput:", OUTPUT_DIR)

Project Root: d:\CodingD\ALPR

Source 1 (Synthetic Special): d:\CodingD\ALPR\data\plate_upper_synth_special
Source 2 (Upper Train): d:\CodingD\ALPR\train_ocr\data\upper_train

Output: d:\CodingD\ALPR\train_ocr\data\upper_train_with_special


In [7]:
# Step 1: Load CSV files
print("Loading CSV files...")
df_synth = pd.read_csv(SYNTH_SPECIAL_CSV, encoding='utf-8-sig')
df_train = pd.read_csv(UPPER_TRAIN_CSV, encoding='utf-8-sig')

print(f"\nSynthetic Special: {len(df_synth):,} samples")
print(f"Upper Train: {len(df_train):,} samples")
print(f"Total: {len(df_synth) + len(df_train):,} samples")

# Display sample data
print("\n--- Synthetic Special (first 3 rows) ---")
print(df_synth.head(3))

print("\n--- Upper Train (first 3 rows) ---")
print(df_train.head(3))

Loading CSV files...

Synthetic Special: 50,000 samples
Upper Train: 158,166 samples
Total: 208,166 samples

--- Synthetic Special (first 3 rows) ---
                         filename      label                   source  \
0  synth_upper_special_000000.jpg  รวยยศ 859  synthetic_upper_special   
1  synth_upper_special_000001.jpg   เฮง 7943  synthetic_upper_special   
2  synth_upper_special_000002.jpg   สวย 4392  synthetic_upper_special   

   transactionDate      plate  province_code  province_description  \
0              NaN  รวยยศ 859            NaN                   NaN   
1              NaN   เฮง 7943            NaN                   NaN   
2              NaN   สวย 4392            NaN                   NaN   

   brand_description  colors_code  colors_description  vehicleClass  
0                NaN          NaN                 NaN           1.0  
1                NaN          NaN                 NaN           1.0  
2                NaN          NaN                 NaN           1.

C:\Users\PC\AppData\Local\Temp\ipykernel_19772\1521541941.py:4: DtypeWarning: Columns (3,5,6,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv(UPPER_TRAIN_CSV, encoding='utf-8-sig')


In [8]:
# Step 2: Concatenate CSV files
print("Merging CSV files...")
df_merged = pd.concat([df_synth, df_train], ignore_index=True)

print(f"\nMerged dataset: {len(df_merged):,} samples")
print(f"Columns: {list(df_merged.columns)}")

# Check for duplicates (optional)
duplicates = df_merged['filename'].duplicated().sum()
print(f"Duplicate filenames: {duplicates}")

if duplicates > 0:
    print("⚠️ Warning: Found duplicate filenames!")
    
print("\n--- Merged dataset (first 5 rows) ---")
print(df_merged.head(5))
print("\n--- Merged dataset (random 5 rows from synth) ---")
print(df_merged[df_merged['source'] == 'synthetic_upper_special'].sample(5))

Merging CSV files...

Merged dataset: 208,166 samples
Columns: ['filename', 'label', 'source', 'transactionDate', 'plate', 'province_code', 'province_description', 'brand_description', 'colors_code', 'colors_description', 'vehicleClass']
Duplicate filenames: 0

--- Merged dataset (first 5 rows) ---
                         filename      label                   source  \
0  synth_upper_special_000000.jpg  รวยยศ 859  synthetic_upper_special   
1  synth_upper_special_000001.jpg   เฮง 7943  synthetic_upper_special   
2  synth_upper_special_000002.jpg   สวย 4392  synthetic_upper_special   
3  synth_upper_special_000003.jpg  ญูใพี๊ 20  synthetic_upper_special   
4  synth_upper_special_000004.jpg   ชะนะ 662  synthetic_upper_special   

  transactionDate      plate province_code province_description  \
0             NaN  รวยยศ 859           NaN                  NaN   
1             NaN   เฮง 7943           NaN                  NaN   
2             NaN   สวย 4392           NaN                  

In [9]:
# Step 3: Save merged CSV
print("Saving merged CSV...")
df_merged.to_csv(OUTPUT_CSV, index=False, encoding='utf-8-sig')
print(f"✓ Saved to: {OUTPUT_CSV}")
print(f"  Total rows: {len(df_merged):,}")

Saving merged CSV...
✓ Saved to: d:\CodingD\ALPR\train_ocr\data\upper_train_with_special\labels.csv
  Total rows: 208,166


In [10]:
# Step 4: Copy image files from both datasets
print("\nCopying image files...")
print("This may take several minutes...\n")

# Copy from synthetic special
print(f"Copying from {SYNTH_SPECIAL_DATA}...")
synth_files = list(SYNTH_SPECIAL_DATA.glob("*.jpg"))
for src_file in tqdm(synth_files, desc="Synthetic Special"):
    dst_file = OUTPUT_DATA / src_file.name
    if not dst_file.exists():
        shutil.copy2(src_file, dst_file)

print(f"✓ Copied {len(synth_files):,} synthetic special images")

# Copy from upper train
print(f"\nCopying from {UPPER_TRAIN_DATA}...")
train_files = list(UPPER_TRAIN_DATA.glob("*.jpg"))
for src_file in tqdm(train_files, desc="Upper Train"):
    dst_file = OUTPUT_DATA / src_file.name
    if not dst_file.exists():
        shutil.copy2(src_file, dst_file)

print(f"✓ Copied {len(train_files):,} upper train images")

# Verify
output_files = list(OUTPUT_DATA.glob("*.jpg"))
print(f"\n✓ Total images in output: {len(output_files):,}")
print(f"✓ Expected: {len(df_merged):,}")
print(f"✓ Match: {len(output_files) == len(df_merged)}")


Copying image files...
This may take several minutes...

Copying from d:\CodingD\ALPR\data\plate_upper_synth_special\data...


Synthetic Special: 100%|██████████| 50000/50000 [00:45<00:00, 1107.79it/s]


✓ Copied 50,000 synthetic special images

Copying from d:\CodingD\ALPR\train_ocr\data\upper_train\data...


Upper Train: 100%|██████████| 158166/158166 [02:50<00:00, 929.39it/s] 


✓ Copied 158,166 upper train images

✓ Total images in output: 208,166
✓ Expected: 208,166
✓ Match: True


In [11]:
# Step 5: Summary and verification
print("\n" + "="*60)
print("MERGE COMPLETE")
print("="*60)

print(f"\nOutput directory: {OUTPUT_DIR}")
print(f"\nDataset statistics:")
print(f"  - Synthetic special plates: {len(df_synth):,} samples")
print(f"  - Real upper plates: {len(df_train):,} samples")
print(f"  - Total merged: {len(df_merged):,} samples")

print(f"\nSource distribution:")
source_counts = df_merged['source'].value_counts()
for source, count in source_counts.items():
    percentage = (count / len(df_merged)) * 100
    print(f"  - {source}: {count:,} ({percentage:.1f}%)")

print(f"\nFiles:")
print(f"  - CSV: {OUTPUT_CSV}")
print(f"  - Images: {OUTPUT_DATA}")
print(f"  - Total images: {len(output_files):,}")

print("\n✓ Dataset ready for training!")


MERGE COMPLETE

Output directory: d:\CodingD\ALPR\train_ocr\data\upper_train_with_special

Dataset statistics:
  - Synthetic special plates: 50,000 samples
  - Real upper plates: 158,166 samples
  - Total merged: 208,166 samples

Source distribution:
  - synthetic_upper: 150,000 (72.1%)
  - synthetic_upper_special: 50,000 (24.0%)
  - 8000: 6,454 (3.1%)
  - camera213: 1,712 (0.8%)

Files:
  - CSV: d:\CodingD\ALPR\train_ocr\data\upper_train_with_special\labels.csv
  - Images: d:\CodingD\ALPR\train_ocr\data\upper_train_with_special\data
  - Total images: 208,166

✓ Dataset ready for training!
